# Bike Movement Aggregates

Builds two analysis-ready datasets summarizing bike pickups and dropoffs per station over time:
- `daily_bike_movement`: `station, date, n_pickups, n_dropoffs`
- `hourly_bike_movement`: `station, datetime, n_pickups, n_dropoffs`

**Station key:** `station` is `stations.short_name`, resolved from `raw_trips.start_station_id` / `end_station_id` using the same identity + crosswalk mapping built in [03_station_id_crosswalk.ipynb](03_station_id_crosswalk.ipynb) (`raw_data/station_id_crosswalk.csv`). This gives the best available match rate (~81.45% of trip rows after the manually-resolved crosswalk entries).

**Scope:** trip rows where either endpoint can't be resolved to a known station are **dropped** from these datasets (not bucketed as "unknown") — a pickup requires a resolved start station, a dropoff requires a resolved end station. A trip missing one endpoint still contributes to the other (e.g. a resolved pickup with an unresolved dropoff still counts as a pickup).

**Counting:** `n_pickups` = trips starting at that station/time (by `started_at`); `n_dropoffs` = trips ending at that station/time (by `ended_at`).

**Output:** written to a new top-level `processed_data/` folder (parallel to `raw_data/`) as parquet files.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Notebook lives in notebooks/, so the project root is one level up.
ROOT = Path.cwd().parent
RAW_DATA = ROOT / "raw_data"

STATIONS_PATH = RAW_DATA / "stations.parquet"
RAW_TRIPS_PATH = RAW_DATA / "raw_trips"
CROSSWALK_PATH = RAW_DATA / "station_id_crosswalk.csv"

PROCESSED_DATA = ROOT / "processed_data"
PROCESSED_DATA.mkdir(exist_ok=True)

DAILY_OUTPUT_PATH = PROCESSED_DATA / "daily_bike_movement.parquet"
HOURLY_OUTPUT_PATH = PROCESSED_DATA / "hourly_bike_movement.parquet"

print(f"Raw trips path: {RAW_TRIPS_PATH} (exists={RAW_TRIPS_PATH.exists()})")
print(f"Stations path: {STATIONS_PATH} (exists={STATIONS_PATH.exists()})")
print(f"Crosswalk path: {CROSSWALK_PATH} (exists={CROSSWALK_PATH.exists()})")
print(f"Processed data folder: {PROCESSED_DATA}")

Raw trips path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\raw_trips (exists=True)
Stations path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\stations.parquet (exists=True)
Crosswalk path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\station_id_crosswalk.csv (exists=True)
Processed data folder: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\processed_data


## Load data

Same schema-override approach as notebooks 02/03: `start_station_id` / `end_station_id` are `int64` in older partitions but become alphanumeric strings from 2021-02 onward, so we open the dataset via `pyarrow.dataset` with an explicit schema forcing both ID columns to `string`.

In [2]:
stations = pd.read_parquet(STATIONS_PATH)

_raw_dataset = ds.dataset(RAW_TRIPS_PATH, format="parquet", partitioning="hive")
_id_string_fields = []
for field in _raw_dataset.schema:
    if field.name in ("start_station_id", "end_station_id"):
        _id_string_fields.append(pa.field(field.name, pa.string()))
    else:
        _id_string_fields.append(field)
_trips_schema = pa.schema(_id_string_fields)

trips_dataset = ds.dataset(RAW_TRIPS_PATH, format="parquet", partitioning="hive", schema=_trips_schema)
trips = trips_dataset.to_table(
    columns=["start_station_id", "end_station_id", "started_at", "ended_at"]
).to_pandas()

trips["started_at"] = pd.to_datetime(trips["started_at"], errors="coerce")
trips["ended_at"] = pd.to_datetime(trips["ended_at"], errors="coerce")

station_ids = set(stations["short_name"].dropna().unique())

print(f"stations: {stations.shape[0]:,} rows")
print(f"trips:    {trips.shape[0]:,} rows")

stations: 866 rows
trips:    55,897,151 rows


## Resolve station IDs using the crosswalk

Build `id_to_short_name`: identity mapping for IDs that already match `stations.short_name` directly, plus `resolved` entries from the notebook 03 crosswalk CSV. Apply it to get `start_resolved` / `end_resolved` columns, then drop rows where the relevant endpoint doesn't resolve (per-endpoint — a trip can still contribute a pickup even if its dropoff is unresolved, and vice versa).

In [3]:
id_to_short_name = {sid: sid for sid in station_ids}

if CROSSWALK_PATH.exists():
    crosswalk = pd.read_csv(CROSSWALK_PATH, dtype={"trip_station_id": str, "chosen_short_name": str})
    resolved = crosswalk.loc[crosswalk["status"] == "resolved"].copy()
    # Validate: only trust chosen_short_name values that are real stations.short_name entries.
    invalid_mask = ~resolved["chosen_short_name"].isin(station_ids)
    if invalid_mask.any():
        print(f"WARNING: ignoring {invalid_mask.sum()} crosswalk row(s) with an invalid chosen_short_name:")
        print(resolved.loc[invalid_mask, ["trip_station_id", "chosen_short_name"]].to_string(index=False))
    resolved = resolved.loc[~invalid_mask]
    id_to_short_name.update(dict(zip(resolved["trip_station_id"], resolved["chosen_short_name"])))
    print(f"Applied {len(resolved):,} resolved crosswalk entries")
else:
    print("No crosswalk CSV found — using identity mapping only (run notebook 03 first for better coverage)")

trips["start_resolved"] = trips["start_station_id"].map(id_to_short_name)
trips["end_resolved"] = trips["end_station_id"].map(id_to_short_name)

total_rows = len(trips)
pickup_resolved = trips["start_resolved"].notna() & trips["started_at"].notna()
dropoff_resolved = trips["end_resolved"].notna() & trips["ended_at"].notna()

print(f"\nTotal trip rows:                         {total_rows:,}")
print(f"Rows contributing a resolved pickup:      {pickup_resolved.sum():,} ({pickup_resolved.mean():.2%})")
print(f"Rows contributing a resolved dropoff:      {dropoff_resolved.sum():,} ({dropoff_resolved.mean():.2%})")

Applied 7 resolved crosswalk entries

Total trip rows:                         55,897,151
Rows contributing a resolved pickup:      49,542,371 (88.63%)
Rows contributing a resolved dropoff:      49,198,810 (88.02%)


## Build hourly aggregates

Floor each resolved pickup/dropoff timestamp to the hour, count separately per `(station, hour)`, then outer-merge so a station-hour with only pickups (or only dropoffs) still gets a row with the other count as 0.

In [4]:
def aggregate_movement(freq: str, datetime_col: str) -> pd.DataFrame:
    """Aggregate resolved pickups/dropoffs into (station, datetime) -> counts, at the given pandas floor frequency."""
    pickups = trips.loc[pickup_resolved, ["start_resolved", "started_at"]].rename(
        columns={"start_resolved": "station", "started_at": "timestamp"}
    )
    pickups[datetime_col] = pickups["timestamp"].dt.floor(freq)
    pickup_counts = pickups.groupby(["station", datetime_col]).size().rename("n_pickups")

    dropoffs = trips.loc[dropoff_resolved, ["end_resolved", "ended_at"]].rename(
        columns={"end_resolved": "station", "ended_at": "timestamp"}
    )
    dropoffs[datetime_col] = dropoffs["timestamp"].dt.floor(freq)
    dropoff_counts = dropoffs.groupby(["station", datetime_col]).size().rename("n_dropoffs")

    combined = pd.concat([pickup_counts, dropoff_counts], axis=1).fillna(0).astype(int)
    return combined.reset_index().sort_values(["station", datetime_col]).reset_index(drop=True)


hourly_bike_movement = aggregate_movement("h", "datetime")

print(f"hourly_bike_movement: {len(hourly_bike_movement):,} rows")
print(f"sum(n_pickups)  = {hourly_bike_movement['n_pickups'].sum():,} (expected {pickup_resolved.sum():,})")
print(f"sum(n_dropoffs) = {hourly_bike_movement['n_dropoffs'].sum():,} (expected {dropoff_resolved.sum():,})")
hourly_bike_movement.head()

hourly_bike_movement: 24,393,570 rows
sum(n_pickups)  = 49,542,371 (expected 49,542,371)
sum(n_dropoffs) = 49,198,810 (expected 49,198,810)


,station,datetime,n_pickups,n_dropoffs
0,30200,2023-02-01 13:00:00,0,2
1,30200,2023-02-01 14:00:00,0,3
2,30200,2023-02-01 15:00:00,1,3
3,30200,2023-02-01 16:00:00,3,4
4,30200,2023-02-01 18:00:00,0,1


## Build daily aggregates

Same approach, floored to the day. Computed independently from the raw resolved events (not by summing the hourly table) so there's no compounding of rounding/edge effects between the two outputs.

In [5]:
daily_bike_movement = aggregate_movement("D", "date")
daily_bike_movement["date"] = daily_bike_movement["date"].dt.date

print(f"daily_bike_movement: {len(daily_bike_movement):,} rows")
print(f"sum(n_pickups)  = {daily_bike_movement['n_pickups'].sum():,} (expected {pickup_resolved.sum():,})")
print(f"sum(n_dropoffs) = {daily_bike_movement['n_dropoffs'].sum():,} (expected {dropoff_resolved.sum():,})")
daily_bike_movement.head()

daily_bike_movement: 2,456,856 rows
sum(n_pickups)  = 49,542,371 (expected 49,542,371)
sum(n_dropoffs) = 49,198,810 (expected 49,198,810)


,station,date,n_pickups,n_dropoffs
0,30200,2023-02-01,4,13
1,30200,2023-02-02,10,14
2,30200,2023-02-03,8,8
3,30200,2023-02-04,4,15
4,30200,2023-02-05,8,7


## Consistency check: hourly summed to daily vs. the daily table

Since both tables were computed independently from raw events, aggregating `hourly_bike_movement` up to daily should match `daily_bike_movement` exactly. Confirms there's no discrepancy between the two granularities.

In [6]:
hourly_rolled_up = hourly_bike_movement.copy()
hourly_rolled_up["date"] = hourly_rolled_up["datetime"].dt.date
hourly_rolled_up = (
    hourly_rolled_up.groupby(["station", "date"])[["n_pickups", "n_dropoffs"]]
    .sum()
    .reset_index()
    .sort_values(["station", "date"])
    .reset_index(drop=True)
)

daily_sorted = daily_bike_movement.sort_values(["station", "date"]).reset_index(drop=True)

pd.testing.assert_frame_equal(hourly_rolled_up, daily_sorted, check_dtype=False)
print("OK: hourly_bike_movement rolled up to daily matches daily_bike_movement exactly")

OK: hourly_bike_movement rolled up to daily matches daily_bike_movement exactly


## Spot check: one station's daily total against a manual count

Pick the highest-volume station and confirm its daily pickup/dropoff totals match a direct filter+count on the raw resolved trip data.

In [7]:
top_station = daily_bike_movement.groupby("station")["n_pickups"].sum().idxmax()
sample_date = daily_bike_movement.loc[daily_bike_movement["station"] == top_station, "date"].iloc[0]

expected_pickups = (
    (trips["start_resolved"] == top_station) & (trips["started_at"].dt.date == sample_date)
).sum()
expected_dropoffs = (
    (trips["end_resolved"] == top_station) & (trips["ended_at"].dt.date == sample_date)
).sum()

row = daily_bike_movement.loc[
    (daily_bike_movement["station"] == top_station) & (daily_bike_movement["date"] == sample_date)
].iloc[0]

print(f"Station: {top_station}, date: {sample_date}")
print(f"daily_bike_movement: n_pickups={row['n_pickups']}, n_dropoffs={row['n_dropoffs']}")
print(f"manual count:        n_pickups={expected_pickups}, n_dropoffs={expected_dropoffs}")

assert row["n_pickups"] == expected_pickups
assert row["n_dropoffs"] == expected_dropoffs
print("\nOK: spot check matches")

Station: 31623, date: 2010-10-26
daily_bike_movement: n_pickups=17, n_dropoffs=9
manual count:        n_pickups=17, n_dropoffs=9

OK: spot check matches


## Save outputs

In [8]:
daily_bike_movement.to_parquet(DAILY_OUTPUT_PATH, index=False)
hourly_bike_movement.to_parquet(HOURLY_OUTPUT_PATH, index=False)

print(f"Saved: {DAILY_OUTPUT_PATH} ({DAILY_OUTPUT_PATH.stat().st_size:,} bytes, {len(daily_bike_movement):,} rows)")
print(f"Saved: {HOURLY_OUTPUT_PATH} ({HOURLY_OUTPUT_PATH.stat().st_size:,} bytes, {len(hourly_bike_movement):,} rows)")

Saved: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\processed_data\daily_bike_movement.parquet (8,875,169 bytes, 2,456,856 rows)
Saved: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\processed_data\hourly_bike_movement.parquet (103,371,772 bytes, 24,393,570 rows)
